# Class 13: APIs and Web Scraping

In this notebook, we will cover two important ways to collect data from the web:

1. **APIs (Application Programming Interfaces)** - A structured way to request data from a server
2. **Web Scraping** - Extracting data directly from web pages using tools like BeautifulSoup

---

## Part 1: Introduction to APIs

An **API** is a set of rules that allows one piece of software to talk to another. When we use an API to get data, we are making an **HTTP request** to a server, and the server sends back a **response** — usually in **JSON** format.

### Key Concepts:
- **Endpoint**: The URL you send your request to
- **Request**: What you ask the server for
- **Response**: What the server sends back
- **JSON (JavaScript Object Notation)**: A lightweight data format that looks a lot like a Python dictionary
- **HTTP Status Codes**: 
  - `200` = OK (success)
  - `404` = Not Found
  - `403` = Forbidden
  - `500` = Server Error

We will use Python's `requests` library to make API calls.

In [ ]:
# Install required libraries if needed
# !pip install requests pandas beautifulsoup4 lxml

In [ ]:
import requests
import json
import pandas as pd
from pandas import json_normalize

print("Libraries loaded successfully!")

---
## Part 2: Connecting to the SEC EDGAR API

The **SEC (Securities and Exchange Commission)** maintains a public database called **EDGAR** (Electronic Data Gathering, Analysis, and Retrieval). It contains financial filings from all publicly traded companies in the United States.

The EDGAR API is **free and requires no API key**, but it does require us to set a `User-Agent` header in our requests to identify ourselves.

### EDGAR API Base URL:
```
https://data.sec.gov/
```

### What we'll do:
1. Look up a company by its **CIK** (Central Index Key) — a unique identifier assigned by the SEC
2. Retrieve the company's filing history
3. Parse the JSON response
4. Load it into a Pandas DataFrame using `json_normalize()`

We'll use **Apple Inc.** as our example. Apple's CIK is `0000320193`.

In [ ]:
# The SEC requires a User-Agent header to identify who is making the request
# Replace the email below with your own email address
headers = {
    "User-Agent": "YourName your_email@example.com"
}

# Apple's CIK number (zero-padded to 10 digits)
cik = "0000320193"

# EDGAR API endpoint for company submissions (filings)
url = f"https://data.sec.gov/submissions/CIK{cik}.json"

print(f"Requesting data from: {url}")

In [ ]:
# Make the API request
response = requests.get(url, headers=headers)

# Check the status code
print(f"Status Code: {response.status_code}")

if response.status_code == 200:
    print("Request successful!")
else:
    print("Something went wrong. Check your URL or headers.")

### Inspecting the JSON Response

The `.json()` method on a `requests` response object automatically parses the JSON into a Python dictionary.

In [ ]:
# Parse the response as JSON
data = response.json()

# What type of object is this?
print(type(data))

# What are the top-level keys?
print("\nTop-level keys:")
print(list(data.keys()))

In [ ]:
# Let's look at some basic company information
print("Company Name:", data.get("name"))
print("CIK:", data.get("cik"))
print("SIC (Industry Code):", data.get("sic"))
print("SIC Description:", data.get("sicDescription"))
print("State of Incorporation:", data.get("stateOfIncorporation"))
print("Fiscal Year End:", data.get("fiscalYearEnd"))

In [ ]:
# The 'filings' key contains the recent filing history
filings = data["filings"]["recent"]

print("Keys inside 'filings > recent':")
print(list(filings.keys()))

### Saving the JSON Response to a File

It's good practice to save raw API responses so you don't have to keep making requests. Let's save the filings data to a JSON file.

In [ ]:
# Save the full response to a JSON file
with open("apple_filings.json", "w") as f:
    json.dump(data, f, indent=4)

print("Data saved to apple_filings.json")

In [ ]:
# We can also read it back in
with open("apple_filings.json", "r") as f:
    loaded_data = json.load(f)

print("Data loaded back from file successfully!")
print("Company:", loaded_data["name"])

---
## Part 3: Loading JSON into Pandas with `json_normalize()`

JSON data is often **nested** — meaning dictionaries inside dictionaries. Pandas provides `json_normalize()` to help flatten this nested structure into a tabular (rows and columns) format.

The `filings["recent"]` object is essentially a dictionary where each key is a column name and each value is a list. This is easy to load directly into a DataFrame.

In [ ]:
# Load the recent filings directly into a DataFrame
filings_df = pd.DataFrame(filings)

print(f"Shape: {filings_df.shape}")
filings_df.head()

In [ ]:
# Now let's use json_normalize() on the top-level data object
# This is especially useful when the JSON has nested fields

# First, let's create a simplified record to normalize
# We'll extract company-level info and normalize it
company_info = {
    "cik": data["cik"],
    "name": data["name"],
    "sic": data["sic"],
    "sicDescription": data["sicDescription"],
    "stateOfIncorporation": data["stateOfIncorporation"],
    "fiscalYearEnd": data["fiscalYearEnd"],
    "addresses": data.get("addresses", {})
}

# json_normalize flattens nested dictionaries
# Notice how nested keys become column names joined by '.'
normalized_df = json_normalize(company_info)

print("Columns after normalization:")
print(list(normalized_df.columns))
normalized_df

In [ ]:
# Let's use json_normalize on the list of filings for a cleaner example
# First, convert the filings dict-of-lists into a list-of-dicts
filings_list = [
    dict(zip(filings.keys(), values))
    for values in zip(*filings.values())
]

# Now normalize it
normalized_filings = json_normalize(filings_list)

print(f"Shape: {normalized_filings.shape}")
print(f"Columns: {list(normalized_filings.columns)}")
normalized_filings.head(10)

In [ ]:
# Let's do some basic analysis on the filings
print("Filing types and counts:")
print(normalized_filings["form"].value_counts().head(10))

In [ ]:
# Filter for just 10-K filings (annual reports)
annual_reports = normalized_filings[normalized_filings["form"] == "10-K"]

print(f"Number of 10-K filings found: {len(annual_reports)}")
annual_reports[["filingDate", "form", "primaryDocument", "primaryDocDescription"]]

In [ ]:
# Let's also look at company facts - financial data reported to the SEC
# This endpoint gives us structured financial data
facts_url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json"

print(f"Requesting financial facts from: {facts_url}")
facts_response = requests.get(facts_url, headers=headers)
print(f"Status Code: {facts_response.status_code}")

In [ ]:
facts_data = facts_response.json()

# Top-level keys
print("Top-level keys:", list(facts_data.keys()))

# The financial data is under 'facts' > 'us-gaap'
gaap_facts = facts_data["facts"]["us-gaap"]
print(f"\nNumber of GAAP financial concepts available: {len(gaap_facts)}")
print("\nSample concepts:")
print(list(gaap_facts.keys())[:20])

In [ ]:
# Let's pull out Net Income data
net_income_data = gaap_facts["NetIncomeLoss"]

print("Label:", net_income_data["label"])
print("Description:", net_income_data["description"][:200])

In [ ]:
# Get the annual (10-K) net income figures
net_income_annual = net_income_data["units"]["USD"]

# Normalize into a DataFrame
net_income_df = json_normalize(net_income_annual)

print(f"Shape: {net_income_df.shape}")
net_income_df.head(10)

In [ ]:
# Filter for annual 10-K filings only
net_income_10k = net_income_df[net_income_df["form"] == "10-K"].copy()

# Convert end date to datetime
net_income_10k["end"] = pd.to_datetime(net_income_10k["end"])

# Sort by date
net_income_10k = net_income_10k.sort_values("end")

# Convert to billions for readability
net_income_10k["val_billions"] = net_income_10k["val"] / 1e9

print("Apple Annual Net Income (in Billions USD):")
net_income_10k[["end", "val_billions", "form", "accn"]].tail(10)

---
## Part 4: Web Scraping with BeautifulSoup

Sometimes the data we want isn't available through an API. In those cases, we can use **web scraping** to extract data directly from a web page's HTML.

### Key Libraries:
- **`requests`**: Fetches the raw HTML of a web page
- **`BeautifulSoup`**: Parses and navigates the HTML structure

### Important Notes on Web Scraping:
- Always check a website's **`robots.txt`** file and **Terms of Service** before scraping
- Be respectful — don't send too many requests too quickly
- APIs are always preferred over scraping when available

### What we'll scrape:
We'll scrape the **Wikipedia page listing the S&P 500 companies**, which is a publicly available table of the 500 largest U.S. publicly traded companies. This is a classic and very useful dataset for business students!

In [ ]:
from bs4 import BeautifulSoup
import time

print("BeautifulSoup imported successfully!")

In [ ]:
# The URL we want to scrape
wiki_url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

# Set a User-Agent so Wikipedia knows who we are
scrape_headers = {
    "User-Agent": "Mozilla/5.0 (Educational purposes) YourName your_email@example.com"
}

# Fetch the page
wiki_response = requests.get(wiki_url, headers=scrape_headers)

print(f"Status Code: {wiki_response.status_code}")
print(f"Content length: {len(wiki_response.text)} characters")

In [ ]:
# Parse the HTML with BeautifulSoup
# 'lxml' is a fast HTML parser
soup = BeautifulSoup(wiki_response.text, "lxml")

# Let's look at the page title to confirm we got the right page
print("Page title:", soup.title.text)

### Understanding HTML Structure

HTML is made up of **tags** like `<table>`, `<tr>` (table row), `<th>` (table header), and `<td>` (table data cell). BeautifulSoup lets us navigate and search this structure.

Common BeautifulSoup methods:
- `soup.find(tag)` — finds the **first** matching tag
- `soup.find_all(tag)` — finds **all** matching tags
- `tag.get_text()` — extracts the text content of a tag
- `tag['attribute']` — gets the value of an HTML attribute (e.g., `tag['href']` for links)

In [ ]:
# Find all tables on the page
tables = soup.find_all("table")
print(f"Number of tables found on the page: {len(tables)}")

In [ ]:
# The S&P 500 table has the id 'constituents'
# We can find it directly using its id attribute
sp500_table = soup.find("table", {"id": "constituents"})

print("Found the S&P 500 table!" if sp500_table else "Table not found.")

In [ ]:
# Extract the header row
# Wikipedia's table may not use a <thead> tag, so we look for the first row containing <th> elements
header_row = sp500_table.find("tr")
headers_row = header_row.find_all("th")
column_names = [header.get_text(strip=True) for header in headers_row]

print("Column names found:")
print(column_names)

In [ ]:
# Extract all data rows from the table body
rows = sp500_table.find("tbody").find_all("tr")

print(f"Number of data rows: {len(rows)}")

# Let's look at the raw HTML of the first row
print("\nFirst row HTML:")
print(rows[0])

In [ ]:
# Parse each row into a list of values
data_rows = []

for row in rows:
    cells = row.find_all("td")
    if cells:  # skip empty rows
        row_data = [cell.get_text(strip=True) for cell in cells]
        data_rows.append(row_data)

print(f"Rows parsed: {len(data_rows)}")
print("\nFirst row of data:")
print(data_rows[0])

In [ ]:
# Load into a Pandas DataFrame
sp500_df = pd.DataFrame(data_rows, columns=column_names)

print(f"Shape: {sp500_df.shape}")
sp500_df.head(10)

In [ ]:
# Let's explore the data
print("Column names:")
print(sp500_df.columns.tolist())

print("\nData types:")
print(sp500_df.dtypes)

In [ ]:
# How many companies are in each sector?
print("Companies per GICS Sector:")
print(sp500_df["GICSSector"].value_counts())

In [ ]:
# Filter for just the Financials sector
financials = sp500_df[sp500_df["GICSSector"] == "Financials"]

print(f"Number of Financial companies in S&P 500: {len(financials)}")
financials[["Symbol", "Security", "GICS Sub-Industry", "Headquarters Location"]].head(10)

In [ ]:
# Save the scraped data to a CSV
sp500_df.to_csv("sp500_companies.csv", index=False)
print("S&P 500 data saved to sp500_companies.csv")

### Scraping Links with BeautifulSoup

In addition to tables, we can also extract **hyperlinks** from a page. Let's extract the Wikipedia links for each S&P 500 company.

In [ ]:
# Extract company name and Wikipedia link from the table
company_links = []

for row in rows:
    cells = row.find_all("td")
    if cells:
        symbol = cells[0].get_text(strip=True)
        # The company name cell contains an anchor tag with the Wikipedia link
        name_cell = cells[1]
        link_tag = name_cell.find("a")
        if link_tag:
            company_name = link_tag.get_text(strip=True)
            wiki_link = "https://en.wikipedia.org" + link_tag.get("href", "")
        else:
            company_name = name_cell.get_text(strip=True)
            wiki_link = None
        company_links.append({"Symbol": symbol, "Company": company_name, "Wikipedia_URL": wiki_link})

links_df = pd.DataFrame(company_links)
print(f"Shape: {links_df.shape}")
links_df.head(10)

---
## Summary

In this notebook, we covered:

| Topic | Tool | Key Takeaway |
|---|---|---|
| Making API requests | `requests` | Use `.get()` with proper headers; check status codes |
| Parsing JSON | `json` | JSON is like a Python dictionary; use `.json()` on a response |
| Saving/loading JSON | `json` | Use `json.dump()` and `json.load()` |
| Flattening JSON to DataFrame | `pandas.json_normalize()` | Handles nested JSON structures automatically |
| Scraping HTML tables | `BeautifulSoup` | Find tags by name, id, or class; extract text with `.get_text()` |
| Extracting links | `BeautifulSoup` | Use `tag['href']` to get URL attributes |

### When to use each approach:
- **Use an API** when one is available — it's more reliable, structured, and respectful of the data provider
- **Use web scraping** when no API exists, but always check the site's Terms of Service first

### Resources:
- [SEC EDGAR API Documentation](https://www.sec.gov/developer)
- [BeautifulSoup Documentation](https://www.crummy.com/software/BeautifulSoup/bs4/doc/)
- [Requests Library Documentation](https://requests.readthedocs.io/en/latest/)
- [Pandas json_normalize Documentation](https://pandas.pydata.org/docs/reference/api/pandas.json_normalize.html)